# Exploring Permutation Invariance

Graphs are a fundamental data structure which model entities and relationships between them. A key property of graphs is that their representation should be invariant to the ordering of nodes. In other words, if we relabel the nodes of a graph, the underlying structure and properties of the graph should remain unchanged.

This notebook explores the concept of permutation invariance in graph representations, particularly focusing on how different neural network architectures handle permutations of node orderings.

In [ ]:
import itertools
import random
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import plotly.express as px
from torch_geometric.nn import GCNConv, global_mean_pool
import math

# reproducibility
seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)


Given a graph, we can represent it using an adjacency matrix. However, the adjacency matrix is sensitive to the ordering of nodes. For example, consider a simple graph with 3 nodes:

```python
A = [[0, 1, 0],
     [1, 0, 1],
     [0, 1, 0]]
```

If we permute the nodes, we get a different adjacency matrix:

```python
B = [[0, 0, 1],
     [0, 0, 1],
     [1, 1, 0]]
```

The total number of permutations of a graph with $n$ nodes is $n!$. For small graphs, we can compute all permutations and see how different models respond to these permutations.
Let's start by generating a random small graph and computing its permutations.

In [ ]:
n = 6
G = nx.barabasi_albert_graph(n, m=2, seed=seed)
A = nx.to_numpy_array(G, dtype=int)  # adjacency as numpy array
nx.draw(G, pos=nx.spring_layout(G, seed=seed), node_color="#9999d1ff", node_size=1000)

# Calculate the number of unique adjacency matrices under all permutations
total_perms = math.factorial(n)            # all permutations of n nodes
unique_adjs = set()
for perm in itertools.permutations(range(n)):
	Aperm = A[np.ix_(perm, perm)]
	unique_adjs.add(Aperm.tobytes())
unique_count = len(unique_adjs)

print(f"n={n}, total_permutations={total_perms}, unique_adjacencies={unique_count}")

# prepare list of permutations for iteration
perms = list(itertools.permutations(range(n)))

## Adjacency Matrices and MLPs

Given an adjacency matrix, the simplest approach to process it is to flatten it into a vector and feed it into a Multi-Layer Perceptron (MLP). 

However, in this format, the MLP has no inherent understanding of the graph structure and is sensitive to node ordering. Let's see how an MLP performs on different permutations of the same graph.

In [ ]:
# 2-layer MLP that takes flattened adjacency matrix -> 1 output
class MLP(nn.Module):
	def __init__(self, in_dim=n*n, hid=16, out_dim=1):
		super().__init__()
		self.net = nn.Sequential(
			nn.Linear(in_dim, hid, bias=False),
			nn.ReLU(),
			nn.Linear(hid, out_dim, bias=False)
		)
	def forward(self, x):
		return self.net(x)

mlp = MLP()
mlp.eval()

# run each permutation through MLP
mlp_outputs = []
with torch.no_grad():
	for perm in perms:
		Aperm = A[np.ix_(perm, perm)]
		x = torch.tensor(Aperm.flatten(), dtype=torch.float32).unsqueeze(0) # [1, n*n]
		out = mlp(x).item()
		mlp_outputs.append(out)
		
def unique_with_tolerance(outputs, tol=1e-8):
    unique_vals = []
    for val in outputs:
        if not any(abs(val - uval) < tol for uval in unique_vals):
            unique_vals.append(val)
    return len(unique_vals)

print("Unique MLP outputs:", unique_with_tolerance(mlp_outputs))

## Adjacency Matrices and CNNs

Another approach is to treat the adjacency matrix as an image, and use Convolutional Neural Networks (CNNs) to process it. CNNs can capture local patterns in the adjacency matrix, but they are still sensitive to node ordering.

In [ ]:
# CNN that takes the adjacency matrix as a 1-channel image and predicts a scalar
class CNN(nn.Module):
    def __init__(self, in_ch=1, hid=16, out_dim=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, hid, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # global spatial pooling
        self.lin = nn.Linear(hid, out_dim, bias=False)

    def forward(self, x):
        x = F.relu(self.conv(x))    
        x = self.pool(x)            # [B, hid, 1, 1]
        x = x.view(x.size(0), -1)   # [B, hid]
        return self.lin(x)          # [B, out_dim]

cnn = CNN()
cnn.eval()

# run each permutation through CNN 
cnn_outputs = []
with torch.no_grad():
    for perm in perms:
        Aperm = A[np.ix_(perm, perm)]
        x_img = torch.tensor(Aperm, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # [1,1,n,n]
        out = cnn(x_img).view(-1).item()
        cnn_outputs.append(out)

print("Unique CNN outputs:", unique_with_tolerance(cnn_outputs))

## Graph Neural Networks (GNNs)

Graph Neural Networks (GNNs) are specifically designed to handle graph-structured data. They operate directly on the graph and aggregate information from neighbouring nodes, making them inherently permutation invariant. Let's see how a simple GNN performs on the same set of permutations.

In [ ]:
# Build a simple 2-layer GNN (GCN) for graph-level regression
class GNN(torch.nn.Module):
	def __init__(self, in_channels=n, hid=16, out_dim=1):
		super().__init__()
		self.conv1 = GCNConv(in_channels, hid)
		self.conv2 = GCNConv(hid, hid)
		self.lin = nn.Linear(hid, out_dim, bias=False)
	def forward(self, x, edge_index, batch):
		x = F.relu(self.conv1(x, edge_index))
		x = F.relu(self.conv2(x, edge_index))
		x = global_mean_pool(x, batch)  # graph-level readout
		return self.lin(x)

gnn = GNN()
gnn.eval()

# node features: identity one-hot (so node identity information is available)
X0 = torch.eye(n)  # [n, n] one-hot node features

# helper to build edge_index from adjacency matrix (both directions)
def edge_index_from_adj(adj):
	rows, cols = np.nonzero(adj)
	edge_index = torch.tensor(np.vstack([rows, cols]), dtype=torch.long)
	return edge_index

# run each permutation through the GNN and save outputs
gnn_outputs = []
with torch.no_grad():
	for perm in perms:
		Aperm = A[np.ix_(perm, perm)]
		Xperm = X0[list(perm), :]  # reorder node features to match permutation
		edge_index = edge_index_from_adj(Aperm)
		# single graph, so batch of zeros
		batch = torch.zeros(Xperm.size(0), dtype=torch.long)
		data_out = gnn(Xperm, edge_index, batch)  # shape [1, out_dim]
		gnn_outputs.append(data_out.view(-1).item())

print("Unique GNN outputs:", unique_with_tolerance(gnn_outputs))


Let's compare the outputs of the MLP, CNN, and GNN on all permutations of the same graph and observe how many unique outputs each model produces. This will illustrate the permutation invariance (or lack thereof) of each architecture.

In [ ]:
idx = list(range(len(perms)))
fig_cnn = go.Figure()
# rotate the default Plotly qualitative color palette by 1
cw = px.colors.qualitative.Plotly
fig_cnn.update_layout(colorway=cw[2:] + cw[:2])
fig_cnn.add_trace(go.Scatter(x=idx, y=mlp_outputs, mode='markers', name='MLP (flattened adj)'))
fig_cnn.add_trace(go.Scatter(x=idx, y=gnn_outputs, mode='markers', name='GNN (permutation equivariant)'))
fig_cnn.add_trace(go.Scatter(x=idx, y=cnn_outputs, mode='markers', name='CNN (adj as image)'))
fig_cnn.update_layout(
    title=f"Model outputs across {len(perms)} permutations",
    xaxis_title="Permutation index",
    yaxis_title="Model output",
    template="plotly_white",
    xaxis=dict(range=[-1, len(perms)])
)
fig_cnn.show()

As we can see from the plot above, the MLP and CNN produce a wide range of outputs for different permutations of the same graph, indicating their sensitivity to node ordering. In contrast, the GNN produces a consistent output across all permutations, demonstrating its permutation invariance.

Of course, any model that maps all input to the same output is trivially permutation invariant. Let's demonstrate using another graph that the GNN can produce different outputs for different graphs while still being permutation invariant.

In [ ]:
# generate a second 5-node BA graph, run through existing models, and plot both graphs' outputs
seed2 = seed + 1
G2 = nx.barabasi_albert_graph(n, m=1, seed=seed2)
A2 = nx.to_numpy_array(G2, dtype=int)
nx.draw(G2, pos=nx.spring_layout(G2, seed=seed2), node_color="#9999d1ff", node_size=1000)

# compute how many unique adjacency matrices under permutations (optional, for title)
unique_adjs2 = set()
for perm in itertools.permutations(range(n)):
    Aperm = A2[np.ix_(perm, perm)]
    unique_adjs2.add(Aperm.tobytes())
unique_count2 = len(unique_adjs2)

# run permutations through existing models (mlp, gnn, cnn)
mlp_outputs2, gnn_outputs2, cnn_outputs2 = [], [], []
with torch.no_grad():
    for perm in perms:
        Aperm = A2[np.ix_(perm, perm)]

        # MLP (flattened adj)
        x2 = torch.tensor(Aperm.flatten(), dtype=torch.float32).unsqueeze(0)
        mlp_outputs2.append(mlp(x2).view(-1).item())

        # GNN (reorder node one-hot features, build edge_index, batch of zeros)
        Xperm = X0[list(perm), :]
        edge_index = edge_index_from_adj(Aperm)
        batch = torch.zeros(Xperm.size(0), dtype=torch.long)
        gnn_out = gnn(Xperm, edge_index, batch)
        gnn_outputs2.append(gnn_out.view(-1).item())

        # CNN (adj as 1-channel image)
        x_img2 = torch.tensor(Aperm, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        cnn_outputs2.append(cnn(x_img2).view(-1).item())

# combined plot: both graphs (original A and new A2) for all models
idx = list(range(len(perms)))
fig_all = go.Figure()
fig_all.add_trace(go.Scatter(x=idx, y=mlp_outputs, mode='markers', name='MLP - graph1'))
fig_all.add_trace(go.Scatter(x=idx, y=mlp_outputs2, mode='markers', name='MLP - graph2'))
fig_all.add_trace(go.Scatter(x=idx, y=gnn_outputs, mode='markers', name='GNN - graph1'))
fig_all.add_trace(go.Scatter(x=idx, y=gnn_outputs2, mode='markers', name='GNN - graph2'))
fig_all.add_trace(go.Scatter(x=idx, y=cnn_outputs, mode='markers', name='CNN - graph1'))
fig_all.add_trace(go.Scatter(x=idx, y=cnn_outputs2, mode='markers', name='CNN - graph2'))

fig_all.update_layout(
    title=f"Model outputs across {len(perms)} permutations — graph1 unique_adjs={unique_count}, graph2 unique_adjs={unique_count2}",
    xaxis_title="Permutation index",
    yaxis_title="Model output",
    template="plotly_white",
    xaxis=dict(range=[-1, len(perms)])
)
fig_all.show()